In [ ]:
PATH_Virus= r"F:\Data_Train_PE\Data_Train\Virus64"
PATH_Clean = r"F:\Data_Train_PE\Data_Train\Clean64"
PATH_ADD_XTest =r"D:\Retrain_clean\Clean64"

In [ ]:
import os
from sklearn.model_selection import train_test_split

clean_files = [os.path.join(PATH_Clean, f) for f in os.listdir(PATH_Clean)]
virus_files = [os.path.join(PATH_Virus, f) for f in os.listdir(PATH_Virus)]
add_test = [os.path.join(PATH_ADD_XTest, f) for f in os.listdir(PATH_ADD_XTest)]
y_clean = [0] * len(clean_files)  # Nhãn 0 cho các tệp clean
y_virus = [1] * len(virus_files)  # Nhãn 1 cho các tệp virus

X_train = clean_files + virus_files
y_train = y_clean + y_virus
# In ra thông tin về số lượng tệp trong tập train và test
print(f"Số lượng file trong tập train: {len(X_train)}")


In [ ]:
from sklearn.model_selection import train_test_split

# Chia tập dữ liệu thành 80% train và 20% test
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
X_train = X_train + add_test
y_train = y_train + [0] * len(add_test)  # Gán nhãn 0 cho các tệp trong add_test


In [ ]:
def splitData(filepath):
    with open(filepath, 'r',encoding ='latin-1') as file:
        # Đọc tất cả dữ liệu trong file
        data_compact = file.read().strip()
        data_split = data_compact.split('\t')  
    return data_split

In [ ]:
import re
def extract_import_raw_tokens(data_split):
    if len(data_split) < 776:
        new_elements = ['0.0'] * 256
        data_split = data_split[:292] + new_elements + data_split[292:]
    data_tail = data_split[776:]
    combined_string = ''.join(data_tail)
    s = re.sub(r'[\n\t]', '', combined_string).lower()
    return s.split()

# Phần chạy cho Net


In [ ]:
import re
def extract_import_raw_tokens(data_split):
    data_tail = data_split[795:]
    combined_string = ''.join(data_tail)
    s = re.sub(r'[\n\t]', '', combined_string).lower()
    return s.split()

In [ ]:
import re

# CRT/toolchain noise: không token hoá chi tiết
CRT_DLL_RE = re.compile(r'^(api-ms-win-|ext-ms-win-|msvcp|vcruntime|ucrtbase)', re.IGNORECASE)

# WinAPI DLL whitelist 
WINAPI_DLLS = {
    'kernel32.dll','ntdll.dll','advapi32.dll','user32.dll','gdi32.dll',
    'shell32.dll','shlwapi.dll','ole32.dll','oleaut32.dll','comctl32.dll',
    'wininet.dll','winhttp.dll','ws2_32.dll','urlmon.dll',
    'crypt32.dll','bcrypt.dll','secur32.dll',
    'iphlpapi.dll','psapi.dll','dbghelp.dll','wtAsapi32.dll'
}

# strip @12, @8 ... thường gặp trong stdcall decorate
DECORATION_RE = re.compile(r'@\d+$')

In [ ]:
X_train[51906]

# Xử lý sections

In [ ]:
# trích xuất phần chứa text section 
import re
def sections(data_split):
  if len(data_split) < 776:
      new_elements = ['0.0'] * 256
      data_split = data_split[:292] + new_elements + data_split[292:]
  data_sections = data_split[775].lower()
  sections = data_sections.split()
  newSection =[]
  for i in range(10):
    if len(sections) < 10:
        if i>=len(sections):
            newSection.append('null')
        else:
            newSection.append(sections[i]) 
    else:
            newSection.append(sections[i])
  return newSection
    
    
    


# Phần chạy cho Net


In [ ]:
import re
def sections(data_split):
  if len(data_split) == 540:
      new_elements = ['0.0'] * 256
      data_split = data_split[:292] + new_elements + data_split[292:]
  data_sections = data_split[794].lower()
  sections = data_sections.split()
  newSection =[]
  for i in range(10):
    if len(sections) < 10:
        if i>=len(sections):
            newSection.append('null')
        else:
            newSection.append(sections[i]) 
    else:
            newSection.append(sections[i])
  return newSection

In [ ]:
# trả về dữ liệu chứa các section,đã được padding
def data_section(listPath):
    list_section =[]
    for path in listPath:
        data_split = splitData(path) # cắt theo dấu \t
        section = sections(data_split) 
        list_section.append(section)
    return list_section

In [ ]:
# # xây từ điển dict1 cho section,đầu vào là data train,trích xuất phần chứa section và thực hiện build,đầu ra là 1 biến từ điển vector_format
# # Từ điển được xây theo tần xuất xuất hiện của section trong bộ dữ liệu, chuỗi nào xuất hiện càng nhiều thì được đánh số càng nhỏ,bắt đầu từ 1
# def buildVocabSection(X_train):
#     list_section =data_section(X_train)
#     setSection=[]
#     count= 0
#     dict1={}
#     for sublis in list_section:
#         for item in sublis:
#             if item !="null" and item not in setSection:
#                 setSection.append(item)
#                 dict1 [item]=count
#             if item in setSection:
#                 dict1[item] = dict1[item] + 1
#     sorted_dict = dict(sorted(dict1.items(), key=lambda x: x[1], reverse=True))
#     print(sorted_dict)
#     vector_format={}
#     count = 1
#     for key,value in sorted_dict.items():
#         vector_format[key]=count
#         count = count + 1
#     vector_format['null']=0
#     return vector_format

In [ ]:
def buildVocabSection(X_train):
    list_section = data_section(X_train)
    
    # Dùng Counter để đếm tần suất, O(1) lookup thay vì list O(n)
    from collections import Counter
    counter = Counter()
    
    for sublist in list_section:
        for item in sublist:
            if item != "null":
                counter[item] += 1
    
    # Sắp xếp theo tần suất giảm dần
    sorted_items = sorted(counter.items(), key=lambda x: x[1], reverse=True)
    
    # Build vector_format
    vector_format = {key: idx for idx, (key, _) in enumerate(sorted_items, start=1)}
    vector_format['null'] = 0
    
    print(dict(sorted_items))  # giữ lại print như code gốc
    return vector_format

In [ ]:
vector_format = buildVocabSection(X_train) 
vector_format['null']=0
print(vector_format)

In [ ]:
vector_format

In [ ]:
with open("PE64Section", "w") as file:
    # Ghi các key ngăn cách nhau bởi dấu cách
    file.write(" ".join(vector_format.keys()))

In [ ]:
def load_token_dict_from_space_file(path, start_index=1):
    """
    File format: token1 token2 token3 ... (space-separated)
    Return: dict {token: id} with id starting from start_index (default 1)
    """
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read().strip()

    if not content:
        return {}

    tokens = content.split()  # split by any whitespace
    # giữ nguyên thứ tự trong file; token trùng thì lấy cái đầu tiên
    vector_format = {}
    idx = start_index
    for t in tokens:
        if t not in vector_format:
            vector_format[t] = idx
            idx += 1
    return vector_format


# Usage
vector_format = load_token_dict_from_space_file("PE64Section", start_index=1)
vector_format['null']=0
print(len(vector_format), list(vector_format.items())[:5])

In [ ]:
print(vector_format)

In [ ]:
vector_format['null']

In [ ]:
list_section_train = data_section(X_train)
list_section_test = data_section(X_test)

In [ ]:
vector_format['null']=0

In [ ]:
# hàm chuyển text sang số,đầu vào là list section được trích ra dạng chuỗi,và bộ từ điển, đầu ra là 1 list chứa các list con 
def getVectorSection(list_section,vector_format):
    vector_section = []
    for line in list_section:
        vector_line = []
        print(line)
        for i in range(10):
            try:
              vector_line.append(vector_format[line[i]])
            except:
              vector_line.append(list(vector_format.values())[-1]+1)
              print(line[i])
              vector_format[line[i]]= list(vector_format.values())[-1]+1
              pass
        vector_section.append(vector_line)
    return vector_section
    

In [ ]:
vector_section_train = getVectorSection(list_section_train,vector_format)
vector_section_test = getVectorSection(list_section_test,vector_format)

In [ ]:
vector_format['null']=0


In [ ]:
len(vector_section_test)

In [ ]:
print(vector_format)

# xử lý import

In [ ]:
def split_dll_tail(token: str):
    """
    token thô thường là:  KERNEL32.dllVirtualAlloc  hoặc  GNSDK_FP.DLL8  hoặc  Qt5Network.dll
    Trả về (dll, tail) với dll kết thúc bằng '.dll' và tail là phần sau.
    """
    t = token.strip()
    # tìm '.dll' gần đầu (dll name)
    idx = t.lower().find('.dll')
    if idx == -1:
        return None, None
    dll = t[:idx+4]
    tail = t[idx+4:]  # phần sau .dll
    return dll.lower(), tail

def normalize_func_tail(tail: str):
    if tail is None:
        return None
    tail = tail.strip()
    if tail == '':
        return None
    # nếu tail toàn số => ordinal
    if tail.isdigit():
        return f'ord_{tail}'
    # loại @12
    tail = DECORATION_RE.sub('', tail)
    return tail.lower()

In [ ]:
# ================================
# Extended API groups for PE64 + .NET
# ================================

# Dynamic loading / resolving
DYNLOAD_APIS = {
    'loadlibrarya','loadlibraryw','loadlibraryexw','loadlibraryexa',
    'getprocaddress',
    'getmodulehandlea','getmodulehandlew',

    # Native loader (PE64 thường dùng)
    'ldrloaddll',
    'ldrgetprocedureaddress'
}

# Memory / injection / native memory ops
MEM_APIS = {
    'virtualalloc','virtualprotect','virtualquery','virtualfree',
    'virtualallocex',

    # Injection core
    'writeprocessmemory',
    'readprocessmemory',
    'createremotethread',
    'openprocess',
    'queueuserapc',

    # Section mapping
    'mapviewoffile',
    'unmapviewoffile',

    # Native API (PE64 malware hay dùng)
    'ntallocatevirtualmemory',
    'ntprotectvirtualmemory',
    'ntwritevirtualmemory',
    'zwallocatevirtualmemory',
    'zwprotectvirtualmemory'
}

# Process / thread enumeration
PROC_ENUM_APIS = {
    'createtoolhelp32snapshot',
    'process32firstw','process32nextw',
    'thread32first','thread32next',

    # Modern / native enumeration
    'ntquerysysteminformation',
    'enumprocesses',
    'k32enumprocesses'
}

# Registry persistence / config
REG_APIS = {
    'regopenkeyexw','regsetvalueexw','regcreatekeyexw',
    'regqueryvalueexw','regdeletevaluew',

    # Extended
    'regdeletekeyexw',
    'regsetkeyvaluew'
}

# Networking hint DLLs (PE64 + .NET)
NET_HINT_DLLS = {
    'wininet.dll','winhttp.dll','ws2_32.dll','urlmon.dll',
    'dnsapi.dll','iphlpapi.dll','secur32.dll',
    'mswsock.dll',
    'shell32.dll'
}

# Crypto hint DLLs (PE64 + .NET)
CRYPTO_HINT_DLLS = {
    'crypt32.dll',
    'bcrypt.dll',
    'advapi32.dll',
    'ncrypt.dll',
    'rsaenh.dll'
}

# Optional: .NET CLR presence hint (không tạo handcrafted mới,
# chỉ để token hoá dll:xxx)
DOTNET_DLLS = {
    'mscoree.dll',
    'clr.dll',
    'mscorlib.dll',
    'system.ni.dll'
}

In [ ]:
import collections
def featurize_imports(raw_tokens):
    """
    raw_tokens: list token thô từ extract_import_raw_tokens()
    Output:
      - tokens: set[str] (Top-K candidates)
      - feats: dict[str,float] handcrafted counts/flags
    """
    tokens = set()
    feats = collections.Counter()

    dlls_seen = set()
    total_imports = 0
    ordinals = 0
    crt_imports = 0

    has_net = 0
    has_crypto = 0

    for tok in raw_tokens:
        dll, tail = split_dll_tail(tok)
        if dll is None:
            continue

        dlls_seen.add(dll)
        func = normalize_func_tail(tail)

        # DLL presence token (trừ CRT nếu bạn muốn)
        # Mình khuyên: vẫn giữ DLL presence cho third-party, nhưng bỏ CRT presence để giảm nhiễu
        if not CRT_DLL_RE.match(dll):
            tokens.add(f'dll:{dll}')

        # Nếu không có func/ordinal thì chỉ DLL presence
        if func is None:
            continue

        total_imports += 1

        # CRT/toolchain: chỉ count, không token hoá chi tiết
        if CRT_DLL_RE.match(dll):
            crt_imports += 1
            continue

        # ordinal?
        if func.startswith('ord_'):
            ordinals += 1

        # Nếu là WinAPI dll thì emit token chi tiết dll!func
        if dll in WINAPI_DLLS:
            tokens.add(f'{dll}!{func}')

            # groups
            if func in DYNLOAD_APIS:
                feats['cnt_dynload'] += 1
            if func in MEM_APIS:
                feats['cnt_mem'] += 1
            if func in PROC_ENUM_APIS:
                feats['cnt_proc_enum'] += 1
            if func in REG_APIS:
                feats['cnt_reg'] += 1

        # dll-based hints (network/crypto)
        if dll in NET_HINT_DLLS:
            has_net = 1
        if dll in CRYPTO_HINT_DLLS:
            has_crypto = 1

    # core counts
    feats['n_imports_total'] = total_imports
    feats['n_dlls_total'] = len(dlls_seen)
    feats['n_ordinals_total'] = ordinals
    feats['cnt_crt_imports'] = crt_imports
    feats['has_net_dll'] = has_net
    feats['has_crypto_dll'] = has_crypto
    feats['ratio_crt'] = (crt_imports / total_imports) if total_imports > 0 else 0.0
    feats['has_dynload_and_mem'] = 1.0 if (feats['cnt_dynload'] > 0 and feats['cnt_mem'] > 0) else 0.0
    feats['has_proc_enum_and_mem'] = 1.0 if (feats['cnt_proc_enum'] > 0 and feats['cnt_mem'] > 0) else 0.0

    return tokens, feats

In [ ]:
import collections

def build_import_docs(listPath):
    docs_tokens = []
    docs_feats = []
    for path in listPath:
        data_split = splitData(path)
        raw = extract_import_raw_tokens(data_split)
        toks, feats = featurize_imports(raw)
        docs_tokens.append(toks)
        docs_feats.append(feats)
    return docs_tokens, docs_feats

docs_train_tokens, docs_train_feats = build_import_docs(X_train)
docs_test_tokens,  docs_test_feats  = build_import_docs(X_test)

In [ ]:
import math

def compute_token_stats(docs_tokens, y):
    """
    docs_tokens: list[set[str]]
    y: array-like 0/1
    returns:
      stats[token] = (a,b,c,d, df, p_mal_given_t, chi2)
        a: #mal has token
        b: #ben has token
        c: #mal no token
        d: #ben no token
    """
    y = list(y)
    N = len(y)
    N_m = sum(1 for v in y if v == 1)
    N_b = N - N_m

    a = collections.Counter()  # malware has t
    b = collections.Counter()  # benign has t
    df = collections.Counter()

    for toks, label in zip(docs_tokens, y):
        for t in toks:
            df[t] += 1
            if label == 1:
                a[t] += 1
            else:
                b[t] += 1

    stats = {}
    for t in df:
        A = a[t]
        B = b[t]
        C = N_m - A
        D = N_b - B

        # chi-square for 2x2 with small smoothing to avoid zero issues
        # chi2 = N*(AD-BC)^2 / ((A+B)(C+D)(A+C)(B+D))
        denom = (A+B)*(C+D)*(A+C)*(B+D)
        if denom == 0:
            chi2 = 0.0
        else:
            chi2 = (N * (A*D - B*C)**2) / denom

        # P(mal|t) with Laplace smoothing
        p_mal_given_t = (A + 1) / (A + B + 2)

        stats[t] = (A,B,C,D, df[t], p_mal_given_t, chi2)

    return stats, N

In [ ]:
def select_topk_tokens(stats, N, malware_ratio,
                       K=5000,
                       min_df_abs=10,
                       min_df_ratio=0.001,
                       max_df_ratio=0.80,
                       delta=0.07,
                       quota_winapi=3500,
                       quota_dll=1000):

    p_mal_min = malware_ratio + delta

    min_df = max(min_df_abs, int(min_df_ratio * N))
    max_df = int(max_df_ratio * N)

    winapi_candidates = []
    dll_candidates = []
    other_candidates = []

    for t, (A,B,C,D,df,p_mal,chi2) in stats.items():

        if df < min_df or df > max_df:
            continue

        if p_mal < p_mal_min:
            continue

        if '!' in t:
            dll_part = t.split('!')[0]
            if dll_part in WINAPI_DLLS:
                winapi_candidates.append((chi2, t))
            else:
                other_candidates.append((chi2, t))
        elif t.startswith('dll:'):
            dll_candidates.append((chi2, t))
        else:
            other_candidates.append((chi2, t))

    # sort
    winapi_candidates.sort(key=lambda x: x[0], reverse=True)
    dll_candidates.sort(key=lambda x: x[0], reverse=True)
    other_candidates.sort(key=lambda x: x[0], reverse=True)

    selected = []
    selected.extend([t for _, t in winapi_candidates[:quota_winapi]])
    selected.extend([t for _, t in dll_candidates[:quota_dll]])

    remain = K - len(selected)
    if remain > 0:
        pool = (
            winapi_candidates[quota_winapi:] +
            dll_candidates[quota_dll:] +
            other_candidates
        )
        for _, t in pool:
            if len(selected) >= K:
                break
            selected.append(t)

    return selected[:K]

In [ ]:
stats, N = compute_token_stats(docs_train_tokens, y_train)
num_malware = len(virus_files)
num_benign = len(clean_files)

N = num_malware + num_benign
malware_ratio = num_malware / N

topk_tokens = select_topk_tokens(stats, N, malware_ratio, K=5000)

# mapping token -> id
token2id = {t:i for i,t in enumerate(topk_tokens)}

with open("PE64DICT", "w", encoding="utf-8") as f:
    for t in topk_tokens:
        f.write(t + "\n")

In [ ]:
with open("PE64DICT","r",encoding="utf-8") as f:
    pe64_tokens = [line.strip() for line in f if line.strip()]
# in xem bao nhiêu phần tử 
print(f"PE64D tokens: {len(pe64_tokens)}")

In [ ]:
def load_token2id_from_lines(path, start_index=0):
    """
    File format: mỗi dòng 1 token
    Return: dict {token: id}, id bắt đầu từ start_index
    """
    token2id = {}
    idx = start_index
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            t = line.strip()
            if not t:
                continue
            if t not in token2id:   # tránh token trùng
                token2id[t] = idx
                idx += 1
    return token2id

# usage
token2id = load_token2id_from_lines("PE64DICT", start_index=0)
print(len(token2id), list(token2id.items())[:5])


In [ ]:
import numpy as np

# Chốt thứ tự handcrafted features (để train/test/C++ giống nhau)
HAND_KEYS = [
    'n_imports_total','n_dlls_total','n_ordinals_total',
    'cnt_crt_imports','ratio_crt',
    'cnt_dynload','cnt_mem','cnt_proc_enum','cnt_reg',
    'has_net_dll','has_crypto_dll',
    'has_dynload_and_mem','has_proc_enum_and_mem'
]

def vectorize_import_features(docs_tokens, docs_feats, token2id, hand_keys=HAND_KEYS):
    K = len(token2id)
    m = len(hand_keys)
    X = np.zeros((len(docs_tokens), K + m), dtype=np.float32)

    for r, (toks, feats) in enumerate(zip(docs_tokens, docs_feats)):
        # Top-K binary
        for t in toks:
            idx = token2id.get(t)
            if idx is not None:
                X[r, idx] = 1.0

        # Handcrafted appended
        base = K
        for j, k in enumerate(hand_keys):
            X[r, base + j] = float(feats.get(k, 0.0))
    return X

Xip_train = vectorize_import_features(docs_train_tokens, docs_train_feats, token2id)
Xip_test  = vectorize_import_features(docs_test_tokens,  docs_test_feats,  token2id)

In [ ]:
vector_import_train = Xip_train.tolist()
vector_import_test = Xip_test.tolist()

In [ ]:
# Lấy dữ liệu số 
def getDataFloat(X):
    datafloat = []
    for file in X:
        data_origin = splitData(file)
        if len(data_origin) == 540:
            new_elements = ['0.0'] * 256
            data_origin = data_origin[:292] + new_elements + data_origin[292:]
        if len(data_origin) < 776:
            break
        datafloat.append(data_origin[:775])
    return datafloat

In [ ]:
# chuyển hết sang float
def convertFloat(datafloat,vector_section,vector_import):
    for i in range(len(datafloat)):
        datafloat[i] = list(map(float, datafloat[i]))
        vector_section[i] = list(map(float, vector_section[i]))
        vector_import[i] = list(map(float, vector_import[i]))
    return datafloat,vector_section,vector_import

In [ ]:
# Gộp 3 vector chuẩn bị đưa vào training
def mergeData(datafloat,vector_section,vector_import):
    for i in range(len(datafloat)):
        datafloat[i].extend(vector_section[i])
        datafloat[i].extend(vector_import[i])
    return datafloat

In [ ]:
datafloat_train=getDataFloat(X_train)
datafloat_test=getDataFloat(X_test)

In [ ]:
len(datafloat_train)

In [ ]:
datafloat_train,vector_section_train,vector_import_train = convertFloat(datafloat_train,vector_section_train,vector_import_train)
datafloat_test,vector_section_test,vector_import_test = convertFloat(datafloat_test,vector_section_test,vector_import_test)

In [ ]:
len(vector_import_train)

In [ ]:
# Dữ liệu train và test
data_train =mergeData(datafloat_train,vector_section_train,vector_import_train)
data_test = mergeData(datafloat_test,vector_section_test,vector_import_test)

In [ ]:
len(data_train)

In [ ]:
import numpy as np
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def to_x(x):
    if isinstance(x, list):
        x = np.vstack([np.asarray(v) for v in x])
    else:
        x = np.asarray(x)
    return x.reshape(x.shape[0], -1).astype(np.float32)

def to_y(y):
    if isinstance(y, list):
        return np.concatenate([np.asarray(v).ravel() for v in y])
    return np.asarray(y).ravel()

dt_train, dt_test = to_x(data_train), to_x(data_test)
label_train, label_test = to_y(y_train), to_y(y_test)

if len(dt_train) != len(label_train) or len(dt_test) != len(label_test):
    raise ValueError(f"Số mẫu và nhãn không khớp: train {dt_train.shape}/{label_train.shape}, test {dt_test.shape}/{label_test.shape}")

if dt_train.shape[1] != dt_test.shape[1]:
    raise ValueError(f"Số feature không khớp: train={dt_train.shape[1]}, test={dt_test.shape[1]}")

clf = lgb.LGBMClassifier(n_estimators=1000, random_state=42, n_jobs=-1)

clf.fit(dt_train, label_train, eval_set=[(dt_test, label_test)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)])

y_pred = clf.predict(dt_test)

print(f'Best iteration: {clf.best_iteration_}')
print(f'Accuracy: {accuracy_score(label_test, y_pred):.4f}')
print(f'Precision: {precision_score(label_test, y_pred, average="macro", zero_division=0):.4f}')
print(f'Recall: {recall_score(label_test, y_pred, average="macro", zero_division=0):.4f}')
print(f'F1 Score: {f1_score(label_test, y_pred, average="macro", zero_division=0):.4f}')

clf.booster_.save_model('CYAIPE64.txt')
print('Đã lưu model: CYAIPE64.txt')

In [ ]:
import os

MAL_DIR   = r"F:\Data_Train_PE\Data_Train\TestMalware64"
CLEAN_DIR = r"F:\Data_Train_PE\Data_Train\TestClean64"

def list_files_non_recursive(folder):
    return [
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if os.path.isfile(os.path.join(folder, f))
    ]

mal_files   = list_files_non_recursive(MAL_DIR)
clean_files = list_files_non_recursive(CLEAN_DIR)

testNewFile =   clean_files
y_test_new  =  [0] * len(clean_files)
#y_test_new  =[0] * len(clean_files)

print("clean:", len(clean_files),"malware:", len(mal_files), "total:", len(testNewFile))

In [ ]:
testNewFile[5232:5239]

In [ ]:
y_test_new[5232:5239]

In [ ]:
list_section_test_new = data_section(testNewFile)
vector_section_test_new = getVectorSection(list_section_test_new,vector_format)
docs_train_tokens_test, docs_train_feats_test = build_import_docs(testNewFile)
Xip_test = vectorize_import_features(docs_train_tokens_test, docs_train_feats_test, token2id)
vector_import_test_new = Xip_test.tolist()
datafloat_test_new = getDataFloat(testNewFile)
datafloat_test_new,vector_section_test_new,vector_import_test_new = convertFloat(datafloat_test_new,vector_section_test_new,vector_import_test_new)
data_test_new1 = mergeData(datafloat_test_new,vector_section_test_new,vector_import_test_new)

In [ ]:
# Dự đoán data_test_new1
dt_test_new1 = to_x(data_test_new1)
y_pred_new1 = clf.predict(dt_test_new1)
# in ra kết quả accuracy, precision, recall, f1 score
accuracy_new1 = accuracy_score(y_test_new, y_pred_new1)
precision_new1 = precision_score(y_test_new, y_pred_new1, average='macro', zero_division=0)
recall_new1 = recall_score(y_test_new, y_pred_new1, average='macro', zero_division=0)
f1_new1 = f1_score(y_test_new, y_pred_new1, average='macro', zero_division=0)
print(f"Accuracy: {accuracy_new1}")
print(f"Precision: {precision_new1}")
print(f"Recall: {recall_new1}")
print(f"F1 Score: {f1_new1}")


In [ ]:
from pathlib import Path
import numpy as np

y_true = np.asarray(y_test_new).ravel()
y_pred = np.asarray(y_pred_new1).ravel()

if not (len(testNewFile) == len(y_true) == len(y_pred)):
    raise ValueError(
        f"Không khớp số lượng: files={len(testNewFile)}, "
        f"labels={len(y_true)}, predictions={len(y_pred)}"
    )

wrong_indices = np.flatnonzero(y_true != y_pred)

print("Tổng số dự đoán sai:", len(wrong_indices))

for idx in wrong_indices:
    print(
        f"[{idx}] {Path(testNewFile[idx]).name} | "
        f"Nhãn thật: {y_true[idx]} | "
        f"Dự đoán: {y_pred[idx]} | "
        f"Đường dẫn: {testNewFile[idx]}"
    )

In [ ]:
y_test_new[]

# kết quả test của PE 64 với 1128 đặc trưng với 343 đặc trưng hàm import
cách lấy hàm :
    1.       if (count_clean - count_virus) >= 200 and count_virus >=0 and count_virus < 20 :
            # if count_virus  >= 1 and count_virus <=3  and count_clean ==0:
                filtered_terms.append(term)
            elif count_virus>20 and count_clean>10*count_virus:
                filtered_terms.append(term)
    2. 
            if (count_virus - count_clean) >= 200 and count_clean >=0 and count_clean < 20 :
            # if count_virus  >= 1 and count_virus <=3  and count_clean ==0:
                filtered_terms.append(term)
            elif count_clean>20 and count_virus>10*count_clean:
                filtered_terms.append(term)      

In [ ]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

clf = lgb.LGBMClassifier(n_estimators=1000, random_state=42)

clf.fit(data_train, y_train, eval_set=[(data_test, y_test)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)])

y_pred = clf.predict(data_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred, average="macro", zero_division=0):.4f}')
print(f'Recall: {recall_score(y_test, y_pred, average="macro", zero_division=0):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred, average="macro", zero_division=0):.4f}')

clf.booster_.save_model('CYAIPE64.txt')

In [ ]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle

# Khởi tạo mô hình LightGBM
clf = lgb.LGBMClassifier(random_state=42)
# Huấn luyện mô hình
clf.fit(data_train, y_train)

# Dự đoán trên tập test
y_pred = clf.predict(data_test)

# Đánh giá các chỉ số
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

# In kết quả
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')

# Lấy mô hình đã huấn luyện từ LightGBM
model = clf.booster_

# Lưu mô hình vào file model.mdl
with open('CYAIPENETNEW.mdl', 'wb') as file:
    pickle.dump(model, file)


In [ ]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle

# Khởi tạo mô hình LightGBM
clf = lgb.LGBMClassifier(random_state=42)

# Huấn luyện mô hình
clf.fit(data_train, y_train)

# Dự đoán trên tập test
y_pred = clf.predict(data_test)

# Đánh giá các chỉ số
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

# In kết quả
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')

# Lấy mô hình đã huấn luyện từ LightGBM
model = clf.booster_

# Lưu mô hình vào file model.mdl
with open('CYAIPE64.mdl', 'wb') as file:
    pickle.dump(model, file)


In [ ]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle

# Khởi tạo mô hình LightGBM
clf = lgb.LGBMClassifier(random_state=42)

# Huấn luyện mô hình
clf.fit(data_train, y_train)

# Dự đoán trên tập test
y_pred = clf.predict(data_test)

# Đánh giá các chỉ số
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

# In kết quả
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')

# Lấy mô hình đã huấn luyện từ LightGBM
model = clf.booster_

# Lưu mô hình vào file model.mdl
with open('CYAIPE64.mdl', 'wb') as file:
    pickle.dump(model, file)


In [ ]:
len(data_train[1])

In [ ]:
data_test_new1

In [ ]:
import numpy as np
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle
MODEL_PATH = r"E:\AV\Train_AI_Python\Data_Preprocessing\CYAIPENETNEW.mdl"

# --- sanity checks ---
X = data_test_new1
y = np.asarray(y_test_new).astype(int)

print("X shape:", getattr(X, "shape", None))
print("y shape:", y.shape, "unique:", np.unique(y))

try:
    # 1) Load model (LightGBM Booster text model)
    with open(MODEL_PATH, "rb") as f:
        model = pickle.load(f)
    print("Loaded LightGBM Booster OK")

    # 2) Predict proba
    # num_iteration=-1 nghĩa là dùng best_iteration nếu có, không thì dùng toàn bộ
    y_prob_new = model.predict(X, num_iteration=-1)
    print("Dự đoán:",y_prob_new)
    # Nếu output là (N,2) (hiếm) thì lấy cột class=1
    y_prob_new = np.asarray(y_prob_new)
    if y_prob_new.ndim == 2 and y_prob_new.shape[1] >= 2:
        y_prob_new = y_prob_new[:, 1]

    # 3) Threshold
    y_pred_new = (y_prob_new >= 0.5).astype(int)
    
except Exception as e:
    print("FAILED to load/predict with lgb.Booster(model_file=...).")
    print("Error:", repr(e))
    raise

# 4) Metrics
accuracy = accuracy_score(y, y_pred_new)
precision = precision_score(y, y_pred_new, average='macro', zero_division=0)
recall = recall_score(y, y_pred_new, average='macro', zero_division=0)
f1 = f1_score(y, y_pred_new, average='macro', zero_division=0)

print(f'Accuracy:  {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1:        {f1:.4f}')

In [ ]:
import numpy as np
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle
MODEL_PATH = r"E:\AV\Train_AI_Python\Data_Preprocessing\CYAIPE32NEW.mdl"

# --- sanity checks ---
X = data_test_new1
y = np.asarray(y_test_new).astype(int)

print("X shape:", getattr(X, "shape", None))
print("y shape:", y.shape, "unique:", np.unique(y))

try:
    # 1) Load model (LightGBM Booster text model)
    with open(MODEL_PATH, "rb") as f:
        model = pickle.load(f)
    print("Loaded LightGBM Booster OK")

    # 2) Predict proba
    # num_iteration=-1 nghĩa là dùng best_iteration nếu có, không thì dùng toàn bộ
    y_prob_new = model.predict(X, num_iteration=-1)

    # Nếu output là (N,2) (hiếm) thì lấy cột class=1
    y_prob_new = np.asarray(y_prob_new)
    if y_prob_new.ndim == 2 and y_prob_new.shape[1] >= 2:
        y_prob_new = y_prob_new[:, 1]

    # 3) Threshold
    y_pred_new = (y_prob_new >= 0.5).astype(int)

except Exception as e:
    print("FAILED to load/predict with lgb.Booster(model_file=...).")
    print("Error:", repr(e))
    raise

# 4) Metrics
accuracy = accuracy_score(y, y_pred_new)
precision = precision_score(y, y_pred_new, average='macro', zero_division=0)
recall = recall_score(y, y_pred_new, average='macro', zero_division=0)
f1 = f1_score(y, y_pred_new, average='macro', zero_division=0)

print(f'Accuracy:  {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1:        {f1:.4f}')

In [ ]:
# Train với dữ liệu chỉ đến section trên bộ test new
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Dự đoán trên tập test
y_pred_new = clf.predict(data_test_new1)

# Đánh giá các chỉ số
accuracy = accuracy_score(y_test_new, y_pred_new)
precision = precision_score(y_test_new, y_pred_new, average='macro')
recall = recall_score(y_test_new, y_pred_new, average='macro')
f1 = f1_score(y_test_new, y_pred_new, average='macro')

# In kết quả
print(f'Accuracy: {accuracy:.4f}')



# Kết quả test PE 32
1
            if (count_virus - count_clean) >= 5000 and count_clean >=0 and count_clean < 50 :
            # if count_virus  >= 1 and count_virus <=3  and count_clean ==0:
                filtered_terms.append(term)
            elif count_clean>20 and count_virus>10*count_clean:
                filtered_terms.append(term)
                
2
            if (count_clean - count_virus) >= 1000 and count_virus >=0 and count_virus < 20 :
            # if count_virus  >= 1 and count_virus <=3  and count_clean ==0:
                filtered_terms.append(term)
            elif count_virus>20 and count_clean>10*count_virus:
                filtered_terms.append(term)

In [ ]:
# Kết quả train của bộ dữ liệu mà phần dữ liệu import kết hợp những tên hàm import  của top 1000 hàm xuất hiện nhiều nhất và các từ phù hợp với điều kiện trên(tức những từ hay xuất hiện nhưng ko quá phổ biến)
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle

# Khởi tạo mô hình LightGBM
clf = lgb.LGBMClassifier(random_state=42)

# Huấn luyện mô hình
clf.fit(data_train, y_train)

# Dự đoán trên tập test
y_pred = clf.predict(data_test)

# Đánh giá các chỉ số
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

# In kết quả
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')

# Lấy mô hình đã huấn luyện từ LightGBM
model = clf.booster_

# Lưu mô hình vào file model.mdl
with open('CYAIPE32.mdl', 'wb') as file:
    pickle.dump(model, file)


In [ ]:
# Kết quả train của bộ dữ liệu mà phần dữ liệu import chỉ lấy top 1000 hàm xuất hiện nhiều nhất
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle

# Khởi tạo mô hình LightGBM
clf = lgb.LGBMClassifier(random_state=42)

# Huấn luyện mô hình
clf.fit(data_train, y_train)

# Dự đoán trên tập test
y_pred = clf.predict(data_test)

# Đánh giá các chỉ số
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

# In kết quả
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')

# Lấy mô hình đã huấn luyện từ LightGBM
model = clf.booster_

# Lưu mô hình vào file model.mdl
with open('CYAIPE32NEW.mdl', 'wb') as file:
    pickle.dump(model, file)


In [ ]:
# Kết quả test của bộ dữ liệu mà phần dữ liệu import kết hợp những tên hàm import  của top 1000 hàm xuất hiện nhiều nhất và các từ phù hợp với điều kiện trên(tức những từ hay xuất hiện nhưng ko quá phổ biến)
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Dự đoán trên tập test
y_pred_new = clf.predict(data_test_new1)

# Đánh giá các chỉ số
accuracy = accuracy_score(y_test_new, y_pred_new)
precision = precision_score(y_test_new, y_pred_new, average='macro')
recall = recall_score(y_test_new, y_pred_new, average='macro')
f1 = f1_score(y_test_new, y_pred_new, average='macro')

# In kết quả
print(f'Accuracy: {accuracy:.4f}')



In [ ]:
# Kết quả test của bộ dữ liệu mà phần dữ liệu import chỉ lấy top 1000 hàm xuất hiện nhiều nhất
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Dự đoán trên tập test
y_pred_new = clf.predict(data_test_new1)

# Đánh giá các chỉ số
accuracy = accuracy_score(y_test_new, y_pred_new)
precision = precision_score(y_test_new, y_pred_new, average='macro')
recall = recall_score(y_test_new, y_pred_new, average='macro')
f1 = f1_score(y_test_new, y_pred_new, average='macro')

# In kết quả
print(f'Accuracy: {accuracy:.4f}')



# xem mức độ quan trọng của từng đặc trưng trong mô hình 


In [ ]:
import numpy as np
import pickle

# Load mô hình đã lưu từ file pickle
with open('CYAIPE64NEW.mdl', 'rb') as file:
    clf = pickle.load(file)

# Lấy mức độ quan trọng của các đặc trưng từ mô hình LightGBM
feature_importances = clf.feature_importance(importance_type='split')  # Hoặc 'gain'

total_importance = np.sum(feature_importances)
dactrung3 = []

# Tính tỷ lệ mức độ quan trọng (phần trăm)
feature_importance_ratios = (feature_importances / total_importance) * 100

# Xử lý tên đặc trưng nếu data_train là list
# số lượng đặc trưng của model đã train
num_features = len(feature_importances)

feature_names = [i for i in range(num_features)]
a = 0
for feature, importance_ratio in zip(feature_names, feature_importance_ratios):
    if round(importance_ratio, 4) > 0.003 and a > 785:
        dactrung3.append(feature)
    a = a + 1

# In từng phần trăm của từng đặc trưng
for feature, importance_ratio in zip(feature_names, feature_importance_ratios):
    print(f"{feature}: {importance_ratio:.5f}%")